# Example 1: MPC Point Feasibility and Loss of Feasibility

This notebook turns the lecture's **double-integrator feasibility example** into an interactive experiment.

We study the constrained finite-horizon MPC problem

$$
x(k+1)=
\begin{bmatrix}
1 & 1 \\
0 & 1
\end{bmatrix}x(k)+
\begin{bmatrix}
0 \\
1
\end{bmatrix}u(k),
\qquad
y(k)=
\begin{bmatrix}
1 & 0
\end{bmatrix}x(k)
$$

subject to

- input constraint: $-0.5 \le u(k) \le 0.5$
- state constraint: $-5 \le x_i(k) \le 5$

The interactive plot focuses on two different notions:

- **Initial feasible set**: points where the finite-horizon optimization problem is feasible at the current time
- **Closed-loop feasible region for a chosen number of MPC steps**: points whose receding-horizon trajectory remains feasible for the selected number of closed-loop iterations

A point can therefore be feasible **now**, yet still be driven to a future state where the MPC optimization becomes infeasible.


In [ ]:
# Optional dependency install (uncomment if needed)
# %pip install numpy matplotlib ipywidgets cvxpy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
import cvxpy as cp

from functools import lru_cache
from IPython.display import display, Markdown

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (8.8, 7.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11


## Model and Feasibility Setup

We use the lecture example

- default horizon: $N = 3$
- default weights: $Q = I$, $R = 10$
- terminal cost: $P = Q$
- terminal constraint: $x_N \in X$ only

The plot colors mean:

- gray: infeasible already at time $k=0$
- orange: feasible at time $k=0$, but loses feasibility within the chosen number of MPC steps
- green: remains feasible for all chosen MPC steps
- black contour: the boundary of the initial feasible set $X_N$

The black feasible boundary depends on the horizon and constraints, but not on the cost weights.


In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])

x_min = np.array([-5.0, -5.0])
x_max = np.array([ 5.0,  5.0])
u_min = -0.5
u_max =  0.5

X_PLOT_MIN, X_PLOT_MAX = -5.0, 5.0
Y_PLOT_MIN, Y_PLOT_MAX = -5.0, 5.0

RESOLUTION_PRESETS = {
    "Fast": {"grid_n": 19, "traj_n": 5},
    "Balanced": {"grid_n": 25, "traj_n": 7},
    "Fine": {"grid_n": 35, "traj_n": 9},
}


def make_weights(q_scale=1.0, r=10.0):
    Q = float(q_scale) * np.eye(2)
    R = np.array([[float(r)]])
    P = Q.copy()
    return Q, R, P


@lru_cache(maxsize=60000)
def solve_mpc_cached(x1, x2, horizon, q_scale, r):
    x0 = np.array([x1, x2], dtype=float)
    Q, R, P = make_weights(q_scale, r)

    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))

    constraints = [X[:, 0] == x0]
    cost = 0

    for k in range(horizon):
        cost += cp.quad_form(X[:, k], Q) + cp.quad_form(U[:, k], R)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k],
            X[:, k] >= x_min,
            X[:, k] <= x_max,
            U[:, k] >= u_min,
            U[:, k] <= u_max,
        ]

    cost += cp.quad_form(X[:, horizon], P)
    constraints += [
        X[:, horizon] >= x_min,
        X[:, horizon] <= x_max,
    ]

    problem = cp.Problem(cp.Minimize(cost), constraints)
    problem.solve(
        solver=cp.OSQP,
        warm_start=True,
        verbose=False,
        eps_abs=1e-5,
        eps_rel=1e-5,
        max_iter=20000,
    )

    if U.value is None:
        return {"feasible": False, "u0": None, "status": problem.status}

    return {
        "feasible": True,
        "u0": float(U.value[0, 0]),
        "status": problem.status,
    }


def solve_mpc_step(x0, horizon, q_scale, r):
    x1 = float(np.round(x0[0], 8))
    x2 = float(np.round(x0[1], 8))
    return solve_mpc_cached(x1, x2, int(horizon), float(q_scale), float(r))


def simulate_closed_loop(x0, horizon, sim_steps, q_scale, r):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []

    for _ in range(sim_steps):
        sol = solve_mpc_step(x, horizon, q_scale, r)
        statuses.append(sol["status"])
        if not sol["feasible"]:
            return {
                "initial_feasible": True,
                "closed_loop_feasible": False,
                "x": np.array(xs),
                "u": np.array(us),
                "fail_state": x.copy(),
                "statuses": statuses,
            }

        u = sol["u0"]
        us.append(u)
        x = A @ x + B[:, 0] * u
        xs.append(x.copy())

        if np.any(x < x_min - 1e-8) or np.any(x > x_max + 1e-8):
            return {
                "initial_feasible": True,
                "closed_loop_feasible": False,
                "x": np.array(xs),
                "u": np.array(us),
                "fail_state": x.copy(),
                "statuses": statuses,
            }

    return {
        "initial_feasible": True,
        "closed_loop_feasible": True,
        "x": np.array(xs),
        "u": np.array(us),
        "fail_state": None,
        "statuses": statuses,
    }


def classify_point(x0, horizon, sim_steps, q_scale, r):
    first = solve_mpc_step(x0, horizon, q_scale, r)
    if not first["feasible"]:
        return 0, {
            "initial_feasible": False,
            "closed_loop_feasible": False,
            "x": np.array([x0], dtype=float),
            "u": np.array([]),
            "fail_state": np.array(x0, dtype=float),
            "statuses": [first["status"]],
        }

    sim = simulate_closed_loop(x0, horizon, sim_steps, q_scale, r)
    return (2 if sim["closed_loop_feasible"] else 1), sim


def evaluate_grid(horizon, sim_steps, q_scale, r, grid_n, progress=None):
    xs = np.linspace(X_PLOT_MIN, X_PLOT_MAX, grid_n)
    ys = np.linspace(Y_PLOT_MIN, Y_PLOT_MAX, grid_n)
    X1, X2 = np.meshgrid(xs, ys)

    closed_loop_class = np.zeros_like(X1, dtype=int)
    initial_feasible_mask = np.zeros_like(X1, dtype=bool)

    total_rows = grid_n
    for i in range(grid_n):
        for j in range(grid_n):
            x0 = np.array([X1[i, j], X2[i, j]])
            first = solve_mpc_step(x0, horizon, q_scale, r)
            if first["feasible"]:
                initial_feasible_mask[i, j] = True
                sim = simulate_closed_loop(x0, horizon, sim_steps, q_scale, r)
                closed_loop_class[i, j] = 2 if sim["closed_loop_feasible"] else 1
            else:
                closed_loop_class[i, j] = 0

        if progress is not None:
            progress("grid", i + 1, total_rows)

    return X1, X2, closed_loop_class, initial_feasible_mask


def generate_sample_trajectories(horizon, sim_steps, q_scale, r, sample_n, progress=None):
    xs = np.linspace(X_PLOT_MIN, X_PLOT_MAX, sample_n)
    ys = np.linspace(Y_PLOT_MIN, Y_PLOT_MAX, sample_n)
    trajectories = []

    total = sample_n * sample_n
    count = 0
    for x1 in xs:
        for x2 in ys:
            x0 = np.array([x1, x2], dtype=float)
            cls, sim = classify_point(x0, horizon, sim_steps, q_scale, r)
            trajectories.append({"class": cls, "sim": sim, "x0": x0})
            count += 1
            if progress is not None:
                progress("traj", count, total)

    return trajectories


def plot_feasibility_map(
    horizon,
    sim_steps,
    q_scale,
    r,
    resolution_name="Balanced",
    selected_point=None,
    progress=None,
):
    preset = RESOLUTION_PRESETS[resolution_name]
    grid_n = preset["grid_n"]
    traj_n = preset["traj_n"]

    X1, X2, closed_loop_class, initial_feasible_mask = evaluate_grid(
        horizon=horizon,
        sim_steps=sim_steps,
        q_scale=q_scale,
        r=r,
        grid_n=grid_n,
        progress=progress,
    )

    fig, ax = plt.subplots(1, 1, figsize=(8.9, 7.3))

    cmap_colors = np.array([
        [0.86, 0.86, 0.86, 1.0],
        [0.98, 0.76, 0.53, 1.0],
        [0.66, 0.86, 0.66, 1.0],
    ])
    ax.contourf(X1, X2, closed_loop_class, levels=[-0.5, 0.5, 1.5, 2.5], colors=cmap_colors)

    boundary = initial_feasible_mask.astype(float)
    ax.contour(X1, X2, boundary, levels=[0.5], colors="black", linewidths=1.8)

    trajectories = generate_sample_trajectories(
        horizon=horizon,
        sim_steps=sim_steps,
        q_scale=q_scale,
        r=r,
        sample_n=traj_n,
        progress=progress,
    )

    for item in trajectories:
        sim = item["sim"]
        xs = sim["x"]
        cls = item["class"]
        if len(xs) <= 1:
            ax.scatter(xs[0, 0], xs[0, 1], s=9, color="#444444", alpha=0.35)
            continue

        color = "#2e7d32" if cls == 2 else "#c96a00"
        ax.plot(xs[:, 0], xs[:, 1], color=color, lw=0.9, alpha=0.35)
        ax.scatter(xs[0, 0], xs[0, 1], s=10, color=color, alpha=0.35)

    if selected_point is not None:
        cls, sim = classify_point(selected_point, horizon, sim_steps, q_scale, r)
        xs = sim["x"]
        highlight_color = "#0055cc" if cls == 2 else "#aa0000" if cls == 1 else "#111111"
        ax.plot(xs[:, 0], xs[:, 1], color=highlight_color, lw=2.6, marker="o", ms=4, zorder=9)
        ax.scatter([selected_point[0]], [selected_point[1]], s=90, marker="*", color=highlight_color, edgecolor="black", zorder=10)

        if cls == 0:
            status_text = "Selected point: infeasible already at k = 0"
        elif cls == 1:
            status_text = "Selected point: feasible initially, then loses feasibility"
        else:
            status_text = f"Selected point: feasible for all {sim_steps} MPC steps"

        ax.text(
            0.02,
            0.98,
            status_text,
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(facecolor="white", alpha=0.94, edgecolor=highlight_color, linewidth=2),
        )

    ax.set_title("Point feasibility under receding-horizon MPC")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_xlim(X_PLOT_MIN, X_PLOT_MAX)
    ax.set_ylim(Y_PLOT_MIN, Y_PLOT_MAX)

    legend_handles = [
        mpatches.Patch(color=cmap_colors[0], label="Infeasible at k = 0"),
        mpatches.Patch(color=cmap_colors[1], label="Feasible now, infeasible later"),
        mpatches.Patch(color=cmap_colors[2], label=f"Feasible for all {sim_steps} MPC steps"),
        plt.Line2D([0], [0], color="black", lw=1.8, label="Initial feasible-set boundary"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", framealpha=0.95)
    ax.grid(True, alpha=0.35)
    plt.show()


sanity_seq = []
x = np.array([-4.0, 3.0])
for _ in range(3):
    sol = solve_mpc_step(x, horizon=3, q_scale=1.0, r=10.0)
    sanity_seq.append((x.copy(), sol["feasible"], sol["u0"]))
    if not sol["feasible"]:
        break
    x = A @ x + B[:, 0] * sol["u0"]

display(Markdown("### Lecture Check"))
display(Markdown(f"`x(0)=[-4,3] -> u0={sanity_seq[0][2]:.3f}`"))
display(Markdown(f"`x(1)=[-1,2.5] -> u0={sanity_seq[1][2]:.3f}`"))
display(Markdown(f"`x(2)=[1.5,2] -> feasible? {solve_mpc_step(np.array([1.5, 2.0]), 3, 1.0, 10.0)['feasible']}`"))


## Interactive Feasibility Viewer

The controls are intentionally compact:

- **Controller**: tune the horizon and quadratic weights
- **View**: choose how many closed-loop MPC steps to test and how detailed the recomputation should be
- **Selected point**: highlight one initial condition and its trajectory

A progress bar is shown while the map is being recomputed.


In [ ]:
q_scale_sl = widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05, description="Q scale", continuous_update=False)
r_sl = widgets.FloatLogSlider(value=10.0, base=10, min=-1, max=2, step=0.05, description="R", continuous_update=False)
horizon_sl = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Horizon", continuous_update=False)
sim_steps_sl = widgets.IntSlider(value=3, min=1, max=10, step=1, description="MPC steps", continuous_update=False)
resolution_dd = widgets.Dropdown(options=["Fast", "Balanced", "Fine"], value="Balanced", description="Detail")

x1_sl = widgets.FloatSlider(value=-4.0, min=-5.0, max=5.0, step=0.25, description="x1(0)", continuous_update=False)
x2_sl = widgets.FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.25, description="x2(0)", continuous_update=False)

progress_label = widgets.HTML("<b>Ready.</b>")
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description="Compute", bar_style="", layout=widgets.Layout(width="420px"))
out = widgets.Output()


def set_progress(stage, current, total):
    if total <= 0:
        return
    if stage == "grid":
        base = 0.0
        span = 75.0
        label = "Evaluating feasibility grid..."
    else:
        base = 75.0
        span = 25.0
        label = "Tracing sample trajectories..."

    progress_bar.value = int(base + span * current / total)
    progress_label.value = f"<b>{label}</b> {current}/{total}"


def finish_progress():
    progress_bar.value = 100
    progress_bar.bar_style = "success"
    progress_label.value = "<b>Done.</b>"


def start_progress():
    progress_bar.value = 0
    progress_bar.bar_style = "info"
    progress_label.value = "<b>Starting recomputation...</b>"


def update_ps11(_=None):
    start_progress()
    out.clear_output(wait=True)
    with out:
        plot_feasibility_map(
            horizon=horizon_sl.value,
            sim_steps=sim_steps_sl.value,
            q_scale=q_scale_sl.value,
            r=r_sl.value,
            resolution_name=resolution_dd.value,
            selected_point=np.array([x1_sl.value, x2_sl.value], dtype=float),
            progress=set_progress,
        )
    finish_progress()


for w in [q_scale_sl, r_sl, horizon_sl, sim_steps_sl, resolution_dd, x1_sl, x2_sl]:
    w.observe(update_ps11, names="value")


controller_box = widgets.VBox([horizon_sl, q_scale_sl, r_sl])
view_box = widgets.VBox([sim_steps_sl, resolution_dd])
point_box = widgets.VBox([x1_sl, x2_sl])

controller_card = widgets.VBox([widgets.HTML("<b>Controller</b>"), controller_box], layout=widgets.Layout(width="250px"))
view_card = widgets.VBox([widgets.HTML("<b>View</b>"), view_box], layout=widgets.Layout(width="220px"))
point_card = widgets.VBox([widgets.HTML("<b>Selected Point</b>"), point_box], layout=widgets.Layout(width="250px"))

top_row = widgets.HBox([controller_card, view_card, point_card])
progress_row = widgets.HBox([progress_bar, progress_label])

display(widgets.VBox([top_row, progress_row, out]))
update_ps11()
